In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
%run "./00_setup.ipynb"


In [ ]:
try:
    engine = get_engine()
    odds_ts = pd.read_sql(text("""
        SELECT o.race_id, o.happyo_time, o.umaban, o.tan_odds, o.ninki
        FROM odds_history.odds_time_series o
        JOIN raw.races r ON o.race_id = r.race_id
        WHERE (r.year || r.month_day)::int >= 20200101
        ORDER BY o.race_id, o.happyo_time, o.umaban
        LIMIT 10000
    """), engine)
    print(f"Loaded {len(odds_ts)} odds time-series records")
except Exception as e:
    print(f"DB error: {e} — generating mock odds data")
    np.random.seed(42)
    n = 5000
    mock_times = []
    m, d = 1, 1
    for i in range(n):
        h = 8 + (i % 10)
        mi = (i * 7) % 60
        mock_times.append(f"{m:02d}{d:02d}{h:02d}{mi:02d}")
        d += 1
        if d > 31:
            d = 1
            m += 1
            if m > 12:
                m = 1
    odds_ts = pd.DataFrame({
        "race_id": [f"R{i//50:04d}" for i in range(n)],
        "happyo_time": mock_times,
        "umaban": [i % 16 + 1 for i in range(n)],
        "tan_odds": np.maximum(1.1, np.random.lognormal(2.0, 0.5, n)),
        "ninki": [i % 16 + 1 for i in range(n)],
    })

In [ ]:

# Group by race+horse and compute dynamics
if len(odds_ts) > 0:
    sample = odds_ts.drop_duplicates(subset=["race_id", "umaban"]).head(200)
    display(sample.head())
else:
    print("No data available")


In [ ]:
from betting.late_money_filter import LateMoneyFilter

lmf = LateMoneyFilter()
print("LateMoneyFilter signal thresholds:")
print("  CANCEL:  odds drop >= 25% in last 3 min")
print("  ADD_CANDIDATE: odds rise >= 30% in last 3 min")
print("\nSample signals:")
for odds_t10, odds_t3 in [(10.0, 7.0), (10.0, 14.0), (5.0, 5.0), (5.0, 0.0)]:
    signal = lmf.check_last_3min(horse_no=1, odds_t10=odds_t10, odds_t3=odds_t3)
    print(f"  odds {odds_t10} → {odds_t3}: {signal.name}")


In [ ]:
print("""
## 結論: オーズ変動分析
- LateMoneyFilter は t-3min 基準で3つのシグナル (CANCEL/ADD_CANDIDATE/NO_ACTION) を出力
- 25%以上の急落 → CANCEL, 30%以上の急騰 → ADD_CANDIDATE
- 実データでのシグナル分布と勝率の関係を確認が必要
""")
